In [6]:
"""
IVANNA CRNN Anti-Dolby — Feature Extraction (Python)
Replica EXACTA de la logica en YamnetClassifier.kt para que el modelo
entrenado aqui reciba, en produccion, tensores con la misma distribucion
que vio en entrenamiento.

Contrato (fijo, no tocar sin actualizar tambien el Kotlin):
  - Sample rate: 16000 Hz mono
    - FFT window: 512 samples, ventaneo Hann
      - Hop length: 160 samples
        - 40 filtros Mel triangulares, 0-8000Hz, formula 2595*log10(1+f/700)
          - 32 frames de tiempo por inferencia
            - log(max(energia, 1e-10)) -- SIN normalizacion adicional
              - Tensor entrada: [32, 40, 1] (sin contar batch)
"""

import numpy as np

SAMPLE_RATE = 16000
FRAME_LENGTH = 512
HOP_LENGTH = 160
N_MELS = 40
TIME_FRAMES = 32
INPUT_LENGTH = (TIME_FRAMES - 1) * HOP_LENGTH + FRAME_LENGTH  # 5472 samples
NUM_CLASSES = 4
CLASS_NAMES = ["Voz", "Musica", "Bajos", "Silencio"]


def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def build_mel_filterbank():
    """Banco de filtros Mel triangulares -- misma logica que buildMelFilterbank() en Kotlin."""
    num_bins = FRAME_LENGTH // 2 + 1
    f_min, f_max = 0.0, SAMPLE_RATE / 2.0

    mel_min = hz_to_mel(f_min)
    mel_max = hz_to_mel(f_max)
    mel_points = np.linspace(mel_min, mel_max, N_MELS + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((FRAME_LENGTH + 1) * hz_points / SAMPLE_RATE).astype(int)

    filters = np.zeros((N_MELS, num_bins), dtype=np.float32)
    for m in range(1, N_MELS + 1):
        f_left, f_center, f_right = bin_points[m - 1], bin_points[m], bin_points[m + 1]

        for k in range(f_left, f_center):
            if 0 <= k < num_bins and f_center > f_left:
                filters[m - 1, k] = (k - f_left) / (f_center - f_left)
        for k in range(f_center, f_right):
            if 0 <= k < num_bins and f_right > f_center:
                filters[m - 1, k] = (f_right - k) / (f_right - f_center)

    return filters


def hann_window(size):
    return 0.5 * (1.0 - np.cos(2.0 * np.pi * np.arange(size) / (size - 1)))


_MEL_FILTERBANK = build_mel_filterbank()
_HANN = hann_window(FRAME_LENGTH).astype(np.float32)


def compute_log_mel_spectrogram(audio: np.ndarray) -> np.ndarray:
    """
    audio: array 1D float32 de al menos INPUT_LENGTH samples @ 16kHz.
    return: [TIME_FRAMES, N_MELS] log-mel-spectrograma.

    Usa np.fft.rfft (FFT real de numpy) en vez de reimplementar
    Cooley-Tukey a mano en Python -- son matematicamente equivalentes
    (ambas son FFT radix-2 sobre una ventana de 512 samples, potencia de
    2), asi que el resultado numerico coincide con el Kotlin dentro de
    error de punto flotante. Si quieres blindarlo al 100%, corre el test
    de validacion mas abajo antes de entrenar con esto.
    """
    result = np.zeros((TIME_FRAMES, N_MELS), dtype=np.float32)

    for t in range(TIME_FRAMES):
        start = t * HOP_LENGTH
        end = start + FRAME_LENGTH
        frame = np.zeros(FRAME_LENGTH, dtype=np.float32)
        available = audio[start:end]
        frame[:len(available)] = available
        frame *= _HANN

        spectrum = np.fft.rfft(frame)  # tamaño FRAME_LENGTH/2 + 1
        power_spectrum = (spectrum.real ** 2 + spectrum.imag ** 2).astype(np.float32)

        mel_energy = _MEL_FILTERBANK @ power_spectrum  # [N_MELS]
        result[t] = np.log(np.maximum(mel_energy, 1e-10))

    return result


def validate_against_kotlin_reference(audio: np.ndarray, reference_csv_path: str):
    """
    IMPORTANTE: antes de entrenar con esto, genera en el teléfono (o con
    una prueba unitaria de YamnetClassifier.kt) el log-mel-spectrograma de
    UN clip de audio conocido, guárdalo como .csv, y compáralo aquí contra
    compute_log_mel_spectrogram() del mismo clip. Si difieren más de ~1e-3
    en promedio, hay un desajuste entre el FFT de numpy y el Cooley-Tukey
    manual del Kotlin que hay que investigar antes de entrenar nada.
    """
    try:
        ref_spectrogram = np.loadtxt(reference_csv_path, delimiter=",")
    except FileNotFoundError:
        print(f"Error: El archivo de referencia '{reference_csv_path}' no fue encontrado.")
        return
    except Exception as e:
        print(f"Error al cargar el archivo CSV de referencia: {e}")
        return

    computed_spectrogram = compute_log_mel_spectrogram(audio)

    if ref_spectrogram.shape != computed_spectrogram.shape:
        print(f"Error: Las formas de los espectrogramas no coinciden.\nReferencia: {ref_spectrogram.shape}, Calculado: {computed_spectrogram.shape}")
        return

    if np.allclose(ref_spectrogram, computed_spectrogram, atol=1e-3):
        print("Los espectrogramas coinciden dentro de la tolerancia de 1e-3.")
    else:
        print("¡ADVERTENCIA! Los espectrogramas NO coinciden dentro de la tolerancia de 1e-3.")
        diff = np.abs(ref_spectrogram - computed_spectrogram)
        max_diff_pos = np.unravel_index(np.argmax(diff), diff.shape)
        print(f"La mayor diferencia es {diff[max_diff_pos]:.4e} en la posición {max_diff_pos}")
        # Optionally, print more differing positions
        # diff_indices = np.where(diff > 1e-3)
        # print(f"Posiciones que difieren significativamente: {list(zip(diff_indices[0], diff_indices[1]))[:5]}...")


if __name__ == "__main__":
    # Sanity check rápido: un tono de 1kHz debería concentrar energía
    # en los filtros Mel correspondientes a esa frecuencia.
    t = np.arange(INPUT_LENGTH) / SAMPLE_RATE
    test_tone = (0.5 * np.sin(2 * np.pi * 1000 * t)).astype(np.float32)
    spec = compute_log_mel_spectrogram(test_tone)
    print("Shape del espectrograma:", spec.shape)  # debe ser (32, 40)
    print("Filtro Mel con mayor energía promedio:", np.argmax(spec.mean(axis=0)))

    # Ejemplo de uso de la función de validación (necesitarás un archivo .npy de referencia)
    # Asegúrate de crear un archivo 'mel_reference.csv' en el mismo directorio
    # o proporciona la ruta completa.
    try:
        validate_against_kotlin_reference(test_tone, "mel_reference.csv")
    except Exception as e:
        print(f"Ocurrió un error durante la validación: {e}")

Shape del espectrograma: (32, 40)
Filtro Mel con mayor energía promedio: 14
Error: El archivo de referencia 'mel_reference.csv' no fue encontrado.
